# Gemma 3 270M From Scratch with SKaiNET (A Standalone Notebook)

A Kotlin re-implementation of [Sebastian Raschka's standalone Gemma 3 notebook](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/12_gemma3/standalone-gemma3-plus-kvcache.ipynb) — same architecture, same KV-cache design, same streaming generation — built **from scratch** on top of [SKaiNET](https://github.com/SKaiNET/SKaiNET)'s core tensor API.

- Only `sk.ainet.core` artifacts are used: tensors, the CPU backend, safetensors I/O, tokenizers, and the Hugging Face downloader. **No** `sk.ainet.transformers` dependency — every layer (RMSNorm, RoPE, grouped-query attention, the transformer block) is written out in this notebook so you can see exactly what Gemma 3 computes.
- KV cache included: keys are cached **unrotated** and re-rotated after concatenation, sliding-window layers trim their cache, and the per-step masks are built for exactly the new query rows.
- Requires a recent Kotlin Notebook / Kotlin Jupyter kernel (Kotlin ≥ 2.3): SKaiNET 0.62.4 is compiled with Kotlin 2.4.

In [20]:
@file:DependsOn("sk.ainet.core:skainet-lang-core:0.62.4")
@file:DependsOn("sk.ainet.core:skainet-backend-cpu:0.62.4")
@file:DependsOn("sk.ainet.core:skainet-io-core:0.62.4")
@file:DependsOn("sk.ainet.core:skainet-io-safetensors:0.62.4")
@file:DependsOn("sk.ainet.core:skainet-frontend-core:0.62.4")
@file:DependsOn("sk.ainet.core:skainet-frontend-text:0.62.4")
@file:DependsOn("sk.ainet.core:skainet-data-source:0.62.4")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core:1.11.0")

In [21]:
import kotlinx.coroutines.CoroutineScope
import kotlinx.coroutines.Dispatchers
import kotlinx.coroutines.future.future
import sk.ainet.context.DirectCpuExecutionContext
import sk.ainet.context.ExecutionContext
import sk.ainet.data.source.DataSourceRequest
import sk.ainet.data.source.JvmDataSourceResolver
import sk.ainet.frontend.tokenizer.TextFrontEnds
import sk.ainet.io.safetensors.SafeTensorsParametersLoader
import sk.ainet.io.safetensors.createRandomAccessSource
import sk.ainet.lang.tensor.Shape
import sk.ainet.lang.tensor.Tensor
import sk.ainet.lang.tensor.dsl.tensor
import sk.ainet.lang.tensor.div
import sk.ainet.lang.tensor.gelu
import sk.ainet.lang.tensor.matmulWeightTransposed
import sk.ainet.lang.tensor.mean
import sk.ainet.lang.tensor.narrow
import sk.ainet.lang.tensor.plus
import sk.ainet.lang.tensor.reshape
import sk.ainet.lang.tensor.sqrt
import sk.ainet.lang.tensor.times
import sk.ainet.lang.tensor.unsqueeze
import sk.ainet.lang.types.DType
import sk.ainet.lang.types.FP32
import sk.ainet.lang.types.Int32
import java.io.File
import kotlin.math.cos
import kotlin.math.pow
import kotlin.math.sin
import kotlin.math.sqrt
import kotlin.random.Random

// Everything in this model is FP32 — one alias keeps the signatures readable.
typealias FTensor = Tensor<FP32, Float>

// One ExecutionContext per inference session. It owns the tensor factory and the
// intra-op Schedule (one coroutine task per core on the JVM).
val ctx = DirectCpuExecutionContext.create()
println("Schedule: ${ctx.schedule.name}")

// One (keys, values) pair per attention layer; keys are stored unrotated.
typealias LayerCache = Pair<FTensor, FTensor>

// Bridge for the suspend APIs (HF download, safetensors loading). Not runBlocking:
// the notebook kernel bundles an older kotlinx-coroutines, while SKaiNET pulls 1.11 —
// runBlocking compiled against 1.11 emits runBlockingK, which the kernel runtime lacks.
// future {}.get() uses symbols that are stable across both versions.
fun <T> blocking(block: suspend CoroutineScope.() -> T): T =
    CoroutineScope(Dispatchers.IO).future { block() }.get()

[SKaiNET] Using standard CPU operations (Vector API not available)
Schedule: coroutines(14)


- This notebook supports both the base model and the instruct model; which one to use can be controlled here:

In [22]:
val useInstructModel = true

&nbsp;
## Two tensor helpers

SKaiNET builds tensors through a small DSL: `tensor<FP32, Float>(ctx, FP32::class) { tensor { shape(...) { init } } }`. We wrap the two fills we need — zeros for norm weights, and a Gaussian for the "fresh model" parameter init (the same role `torch.manual_seed(123)` + the default `nn.Linear` init plays in the PyTorch notebook).

> Note: we draw the Gaussian through `randomInit` with `1 - nextDouble()` instead of the built-in `randn()`. The stock Box-Muller uses `ln(u1)` with `u1 ∈ [0, 1)`, and across this model's ~436M parameter draws a `u1 == 0` (→ `Inf`/`NaN`) is near-certain.

In [23]:
val rng = Random(123) // one shared generator = reproducible init, like torch.manual_seed(123)

fun zerosTensor(ctx: ExecutionContext, vararg dims: Int): FTensor =
    tensor<FP32, Float>(ctx, FP32::class) { tensor { shape(*dims) { zeros() } } }

fun randnTensor(ctx: ExecutionContext, vararg dims: Int, std: Float = 0.02f): FTensor =
    tensor<FP32, Float>(ctx, FP32::class) {
        tensor {
            shape(*dims) {
                randomInit({ r ->
                    (kotlin.math.sqrt(-2.0 * kotlin.math.ln(1.0 - r.nextDouble())) *
                        kotlin.math.cos(2.0 * kotlin.math.PI * r.nextDouble()) * std).toFloat()
                }, rng)
            }
        }
    }

&nbsp;
## RoPE — rotary position embeddings with two bases

Gemma 3's signature trick: **sliding-window layers rotate with `theta = 10,000`, full-attention layers with `theta = 1,000,000`**. We precompute one `(cos, sin)` table per base — these are the model's `cos_local` / `sin_local` / `cos_global` / `sin_global` buffers, shaped `[contextLength, headDim]` with the angle table laid out as `[angles | angles]`.

`applyRope` is where the KV cache meets RoPE: `offset` says which absolute position row 0 of `x` sits at. Queries rotate at `startPos`; cached keys are stored **unrotated** and re-rotated from the cache's first position every step. A `[seqLen, headDim]` slice of the table broadcasts over `[batch, heads, seqLen, headDim]` with no reshaping — SKaiNET's elementwise ops align trailing dimensions, NumPy-style.

In [24]:
fun computeRopeParams(
    ctx: ExecutionContext,
    headDim: Int,
    thetaBase: Double,
    contextLength: Int,
): Pair<FTensor, FTensor> {
    require(headDim % 2 == 0) { "Head dimension must be even" }
    val half = headDim / 2
    val invFreq = DoubleArray(half) { i -> 1.0 / thetaBase.pow(2.0 * i / headDim) }
    // angles[pos, d] = pos * invFreq[d % half]  (the table is [angles | angles])
    val cosT = tensor<FP32, Float>(ctx, FP32::class) {
        tensor { shape(contextLength, headDim) { init { idx -> cos(idx[0] * invFreq[idx[1] % half]).toFloat() } } }
    }
    val sinT = tensor<FP32, Float>(ctx, FP32::class) {
        tensor { shape(contextLength, headDim) { init { idx -> sin(idx[0] * invFreq[idx[1] % half]).toFloat() } } }
    }
    return cosT to sinT
}

fun applyRope(x: FTensor, cosT: FTensor, sinT: FTensor, offset: Int = 0): FTensor {
    val seqLen = x.shape[2]
    val headDim = x.shape[3]
    val half = headDim / 2

    val x1 = x.narrow(dim = 3, start = 0, length = half)
    val x2 = x.narrow(dim = 3, start = half, length = half)
    val rotated = x.ops.concat(listOf(x2 * -1f, x1), dim = 3)

    // [seqLen, headDim] broadcasts over [b, nHeads, seqLen, headDim] (trailing-aligned)
    val cosS = cosT.narrow(dim = 0, start = offset, length = seqLen)
    val sinS = sinT.narrow(dim = 0, start = offset, length = seqLen)

    return x * cosS + rotated * sinS
}

&nbsp;
## The config

Declared up front so every layer class below can take it as a constructor parameter. Layers 5, 11 and 17 (every sixth) are full-attention; the rest use the 512-token sliding window.

In [25]:
enum class AttnType { SLIDING, FULL }

data class GemmaConfig(
    val vocabSize: Int,
    val contextLength: Int,
    val embDim: Int,
    val nHeads: Int,
    val nLayers: Int,
    val hiddenDim: Int,
    val headDim: Int,
    val qkNorm: Boolean,
    val nKvGroups: Int,
    val ropeLocalBase: Double,
    val ropeBase: Double,
    val slidingWindow: Int,
    val layerTypes: List<AttnType>,
    val queryPreAttnScalar: Float,
)

val GEMMA3_CONFIG_270M = GemmaConfig(
    vocabSize = 262_144,
    contextLength = 32_768,
    embDim = 640,
    nHeads = 4,
    nLayers = 18,
    hiddenDim = 2048,
    headDim = 256,
    qkNorm = true,
    nKvGroups = 1,
    ropeLocalBase = 10_000.0,
    ropeBase = 1_000_000.0,
    slidingWindow = 512,
    // 5 x sliding, then 1 x full — repeated three times
    layerTypes = List(18) { i -> if ((i + 1) % 6 == 0) AttnType.FULL else AttnType.SLIDING },
    queryPreAttnScalar = 256f,
)

&nbsp;
## The KV cache

Deliberately boring, exactly like the original: one `(keys, values)` slot per layer. All the intelligence (what to cache, when to trim) lives in the blocks.

In [26]:
class KVCache(nLayers: Int) {
    private val cache = arrayOfNulls<LayerCache>(nLayers)
    fun get(layerIdx: Int): LayerCache? = cache[layerIdx]
    fun update(layerIdx: Int, value: LayerCache) { cache[layerIdx] = value }
    fun reset() = cache.indices.forEach { cache[it] = null }
}

&nbsp;
## The architecture — one cell, five classes

Everything below lives in a **single cell** on purpose. Current Kotlin Notebook kernels
(kotlin-jupyter ≤ 0.19) miscompile a class split across cells when its *only* cross-cell
reference is constructing another cell's class (the compiler then fails with
`No mapping for symbol ... INSTANCE_RECEIVER` while generating the constructor).
Functions, enums, data classes and script-level code are unaffected — which is why the
helpers, RoPE tables and config above keep their own cells. One compilation unit for the
class graph sidesteps the bug entirely.

### RMSNorm

Gemma 3 stores **zero-centered** norm weights and applies `(1 + weight)` in the forward pass — a freshly initialized norm (all zeros) is the identity scale. The whole model runs in FP32 here, so unlike the PyTorch version there is no dtype round-trip to manage.

The reduction keeps broadcasting honest: `mean(dim = -1)` drops the last axis, `unsqueeze` puts it back as size 1, and the division broadcasts `[..., 1]` over `[..., embDim]`.

### Linear and the GeGLU feed-forward

Weights stay in the Hugging Face layout `[outFeatures, inFeatures]`, and the forward pass is `matmulWeightTransposed` — so checkpoint tensors drop in with **no transposing** at load time. SKaiNET's `gelu()` defaults to the tanh approximation, which is exactly the flavor Gemma uses (`gelu(x, approximate="tanh")` in PyTorch).

### Grouped-query attention with a KV cache

The heart of the notebook. Three things to notice:

1. **The cache stores unrotated keys.** New K/V are concatenated onto the cache *before* RoPE; the full key tensor is then re-rotated from the cache's first absolute position (`startPos - prevLen`). This is what makes sliding-window trimming correct — positions stay absolute no matter how much history has been dropped.
2. **No `repeat_interleave`.** SKaiNET's fused `scaledDotProductAttention` supports grouped-query attention natively: K/V stay at `nKvGroups` heads and query head `h` reads K/V head `h / groupSize` inside the kernel. The PyTorch notebook has to tile K/V up to the query head count by hand.
3. **Query scaling** uses `queryPreAttnScalar` (256), not `headDim` — for the 270M model they happen to be equal, but the config keeps them distinct, exactly like the original. The scale folds into the fused kernel: `softmax(Q Kᵀ · scale + mask) V`.

### The transformer block — sandwich norms, sliding vs. full attention

Gemma 3 normalizes **four** times per block (before and after both the attention and the feed-forward sub-layers). Two cache-aware details:

- A sliding layer's mask must be as wide as its (trimmed) K length, so it takes the **last `effKvLen` columns** of the local mask.
- After attention, a sliding layer's cache is cut back to the final `slidingWindow` entries — the far past can never be attended to again, so there is no point keeping it.

### The model

The embedding is scaled by `sqrt(embDim)` (a Gemma convention), the four RoPE tables are built once as reusable buffers, and `currentPos` tracks where the KV cache ends.

`createMasks` builds **additive** masks (`0` = attend, `-1e9` = blocked) for just the new query rows `[posStart, posEnd)` over all `posEnd` key columns — during cached decoding that is a single row. The two patterns, for `slidingWindow = 4`:

```text
mask_global (future masked: j > i)        mask_local (future OR far past:
                                           j > i  OR  i - j >= 4)
    j:  0 1 2 3 4 5 6 7                       j:  0 1 2 3 4 5 6 7
 i                                         i
 0:     0 1 1 1 1 1 1 1                    0:     0 1 1 1 1 1 1 1
 1:     0 0 1 1 1 1 1 1                    1:     0 0 1 1 1 1 1 1
 2:     0 0 0 1 1 1 1 1                    2:     0 0 0 1 1 1 1 1
 3:     0 0 0 0 1 1 1 1                    3:     0 0 0 0 1 1 1 1
 4:     0 0 0 0 0 1 1 1                    4:     1 0 0 0 0 1 1 1
 5:     0 0 0 0 0 0 1 1                    5:     1 1 0 0 0 0 1 1
 6:     0 0 0 0 0 0 0 1                    6:     1 1 1 0 0 0 0 1
 7:     0 0 0 0 0 0 0 0                    7:     1 1 1 1 0 0 0 0
```

Where the PyTorch original assembles these from `triu` tricks, a Kotlin lambda can just *state* the predicate.

In [8]:
// ──────────────────────── RMSNorm ────────────────────────

class RMSNorm(ctx: ExecutionContext, val embDim: Int, val eps: Float = 1e-6f) {
    var scale: FTensor = zerosTensor(ctx, embDim)

    fun forward(x: FTensor): FTensor {
        val variance = (x * x).mean(dim = -1).unsqueeze(x.rank - 1) // [..., 1]
        val xNorm = x / (variance + eps).sqrt()
        return xNorm * (1f + scale)
    }
}

// ──────────────────────── Linear & GeGLU feed-forward ────────────────────────

class Linear(ctx: ExecutionContext, val inFeatures: Int, val outFeatures: Int) {
    var weight: FTensor = randnTensor(ctx, outFeatures, inFeatures) // HF layout [out, in]
    fun forward(x: FTensor): FTensor = x.matmulWeightTransposed(weight)
}

// GeGLU: gelu(fc1(x)) * fc2(x) -> fc3
class FeedForward(ctx: ExecutionContext, cfg: GemmaConfig) {
    val fc1 = Linear(ctx, cfg.embDim, cfg.hiddenDim)
    val fc2 = Linear(ctx, cfg.embDim, cfg.hiddenDim)
    val fc3 = Linear(ctx, cfg.hiddenDim, cfg.embDim)

    fun forward(x: FTensor): FTensor =
        fc3.forward(fc1.forward(x).gelu() * fc2.forward(x))
}

// ──────────────────────── Grouped-query attention ────────────────────────

class GroupedQueryAttention(ctx: ExecutionContext, cfg: GemmaConfig) {
    val numHeads = cfg.nHeads
    val numKvGroups = cfg.nKvGroups
    val headDim = cfg.headDim
    val dOut = numHeads * headDim

    val wQuery = Linear(ctx, cfg.embDim, dOut)
    val wKey = Linear(ctx, cfg.embDim, numKvGroups * headDim)
    val wValue = Linear(ctx, cfg.embDim, numKvGroups * headDim)
    val outProj = Linear(ctx, dOut, cfg.embDim)

    val qNorm: RMSNorm? = if (cfg.qkNorm) RMSNorm(ctx, headDim) else null
    val kNorm: RMSNorm? = if (cfg.qkNorm) RMSNorm(ctx, headDim) else null

    val scaling = 1f / sqrt(cfg.queryPreAttnScalar)

    fun forward(
        x: FTensor,
        mask: FTensor,
        cosT: FTensor,
        sinT: FTensor,
        startPos: Int = 0,
        cache: LayerCache? = null,
    ): Pair<FTensor, LayerCache> {
        val b = x.shape[0]
        val numTokens = x.shape[1]
        val ops = x.ops

        // Project and reshape to [b, heads, tokens, headDim]
        var queries = ops.permute(
            wQuery.forward(x).reshape(Shape(b, numTokens, numHeads, headDim)), intArrayOf(0, 2, 1, 3)
        )
        var keysNew = ops.permute(
            wKey.forward(x).reshape(Shape(b, numTokens, numKvGroups, headDim)), intArrayOf(0, 2, 1, 3)
        )
        val valuesNew = ops.permute(
            wValue.forward(x).reshape(Shape(b, numTokens, numKvGroups, headDim)), intArrayOf(0, 2, 1, 3)
        )

        // Optional Q/K normalization (applied to the raw, unrotated tensors)
        qNorm?.let { queries = it.forward(queries) }
        kNorm?.let { keysNew = it.forward(keysNew) }

        // Keep unrotated keys in the cache; rotate after concatenation
        val prevLen = cache?.first?.shape?.get(2) ?: 0
        val keysCat = if (cache != null) ops.concat(listOf(cache.first, keysNew), dim = 2) else keysNew
        val valuesCat = if (cache != null) ops.concat(listOf(cache.second, valuesNew), dim = 2) else valuesNew

        // RoPE: queries at absolute startPos; keys re-rotated from the cache start
        queries = applyRope(queries, cosT, sinT, offset = startPos)
        val keys = applyRope(keysCat, cosT, sinT, offset = startPos - prevLen)

        // Fused attention with native grouped-query support: K/V stay at nKvGroups heads
        val context = ops.scaledDotProductAttention(queries, keys, valuesCat, mask, scale = scaling)

        val out = outProj.forward(
            ops.permute(context, intArrayOf(0, 2, 1, 3)).reshape(Shape(b, numTokens, dOut))
        )
        return out to (keysCat to valuesCat)
    }
}

// ──────────────────────── Transformer block ────────────────────────

class TransformerBlock(ctx: ExecutionContext, val cfg: GemmaConfig, val attnType: AttnType) {
    val att = GroupedQueryAttention(ctx, cfg)
    val ff = FeedForward(ctx, cfg)
    val inputLayernorm = RMSNorm(ctx, cfg.embDim)
    val postAttentionLayernorm = RMSNorm(ctx, cfg.embDim)
    val preFeedforwardLayernorm = RMSNorm(ctx, cfg.embDim)
    val postFeedforwardLayernorm = RMSNorm(ctx, cfg.embDim)

    fun forward(
        x: FTensor,
        maskGlobal: FTensor,
        maskLocal: FTensor,
        cosGlobal: FTensor,
        sinGlobal: FTensor,
        cosLocal: FTensor,
        sinLocal: FTensor,
        startPos: Int = 0,
        cache: LayerCache? = null,
    ): Pair<FTensor, LayerCache> {
        // Attention block with shortcut connection
        var shortcut = x
        var h = inputLayernorm.forward(x)

        val (mask, cosT, sinT) = if (attnType == AttnType.SLIDING) {
            // Take the last effKvLen columns so the mask width equals the K length
            val effKvLen = (cache?.first?.shape?.get(2) ?: 0) + h.shape[1]
            val cols = maskLocal.shape[3]
            Triple(maskLocal.narrow(dim = 3, start = cols - effKvLen, length = effKvLen), cosLocal, sinLocal)
        } else {
            Triple(maskGlobal, cosGlobal, sinGlobal)
        }

        val (attnRaw, newCache) = att.forward(h, mask, cosT, sinT, startPos, cache)

        // Sliding-window layers only ever need the last `slidingWindow` KV entries
        var nextCache = newCache
        if (attnType == AttnType.SLIDING && newCache.first.shape[2] > cfg.slidingWindow) {
            val (k, v) = newCache
            val len = k.shape[2]
            nextCache = k.narrow(2, len - cfg.slidingWindow, cfg.slidingWindow) to
                v.narrow(2, len - cfg.slidingWindow, cfg.slidingWindow)
        }

        h = shortcut + postAttentionLayernorm.forward(attnRaw)

        // Feed-forward block with shortcut connection
        shortcut = h
        val ffn = postFeedforwardLayernorm.forward(ff.forward(preFeedforwardLayernorm.forward(h)))
        return (shortcut + ffn) to nextCache
    }
}

// ──────────────────────── The model ────────────────────────

class Gemma3Model(val ctx: ExecutionContext, val cfg: GemmaConfig) {
    init { require(cfg.layerTypes.size == cfg.nLayers) }

    var tokEmb: FTensor = randnTensor(ctx, cfg.vocabSize, cfg.embDim)
    val blocks = cfg.layerTypes.map { TransformerBlock(ctx, cfg, it) }
    val finalNorm = RMSNorm(ctx, cfg.embDim)
    val outHead = Linear(ctx, cfg.embDim, cfg.vocabSize)

    // Reusable RoPE tables ("buffers"): sliding layers use theta=10k, full layers theta=1M
    val ropeLocal = computeRopeParams(ctx, cfg.headDim, cfg.ropeLocalBase, cfg.contextLength)
    val ropeGlobal = computeRopeParams(ctx, cfg.headDim, cfg.ropeBase, cfg.contextLength)

    var currentPos = 0 // tracks the KV-cache position

    /** Additive masks for query rows [posStart, posEnd): 0 = attend, -1e9 = blocked. */
    fun createMasks(posStart: Int, posEnd: Int): Pair<FTensor, FTensor> {
        val rows = posEnd - posStart
        fun mask(blocked: (i: Int, j: Int) -> Boolean) = tensor<FP32, Float>(ctx, FP32::class) {
            tensor {
                shape(1, 1, rows, posEnd) {
                    init { idx -> if (blocked(posStart + idx[2], idx[3])) -1e9f else 0f }
                }
            }
        }
        val maskGlobal = mask { i, j -> j > i }
        val maskLocal = mask { i, j -> j > i || i - j >= cfg.slidingWindow }
        return maskGlobal to maskLocal
    }

    fun forward(inputIds: IntArray, cache: KVCache? = null): FTensor {
        val seqLen = inputIds.size
        val ops = tokEmb.ops

        // Token embedding, scaled by sqrt(embDim)
        val ids = tensor<Int32, Int>(ctx, Int32::class) { tensor { shape(seqLen) { fromArray(inputIds) } } }
        @Suppress("UNCHECKED_CAST")
        var x = ops.indexSelect(tokEmb, ids as Tensor<DType, *>, dim = 0)
            .reshape(Shape(1, seqLen, cfg.embDim)) * sqrt(cfg.embDim.toFloat())

        val posStart: Int
        if (cache != null) {
            posStart = currentPos
            currentPos += seqLen
        } else {
            posStart = 0
        }
        val (maskGlobal, maskLocal) = createMasks(posStart, posStart + seqLen)

        for ((i, block) in blocks.withIndex()) {
            val (h, newCache) = block.forward(
                x,
                maskGlobal = maskGlobal,
                maskLocal = maskLocal,
                cosGlobal = ropeGlobal.first,
                sinGlobal = ropeGlobal.second,
                cosLocal = ropeLocal.first,
                sinLocal = ropeLocal.second,
                startPos = posStart,
                cache = cache?.get(i),
            )
            x = h
            cache?.update(i, newCache)
        }

        return outHead.forward(finalNorm.forward(x))
    }

    fun resetKvCache() { currentPos = 0 }
}

&nbsp;
## Build the model

(This allocates ~436M FP32 parameters plus four RoPE tables — about 1.9 GB. Give the kernel JVM a few GB of heap.)

In [9]:
val model = Gemma3Model(ctx, GEMMA3_CONFIG_270M)

An homage to the PyTorch notebook's `Out[12]` — the printed module tree:

In [10]:
fun Gemma3Model.summary(): String {
    fun linear(l: Linear) = "Linear(inFeatures=${l.inFeatures}, outFeatures=${l.outFeatures}, bias=false)"
    val blk = blocks.first()
    return buildString {
        appendLine("Gemma3Model(")
        appendLine("  (tok_emb): Embedding(${cfg.vocabSize}, ${cfg.embDim})")
        appendLine("  (blocks): ${cfg.nLayers} x TransformerBlock(")
        appendLine("    (att): GroupedQueryAttention(")
        appendLine("      (W_query): ${linear(blk.att.wQuery)}")
        appendLine("      (W_key): ${linear(blk.att.wKey)}")
        appendLine("      (W_value): ${linear(blk.att.wValue)}")
        appendLine("      (out_proj): ${linear(blk.att.outProj)}")
        appendLine("      (q_norm): RMSNorm(${blk.att.qNorm?.embDim})")
        appendLine("      (k_norm): RMSNorm(${blk.att.kNorm?.embDim})")
        appendLine("    )")
        appendLine("    (ff): FeedForward(")
        appendLine("      (fc1): ${linear(blk.ff.fc1)}")
        appendLine("      (fc2): ${linear(blk.ff.fc2)}")
        appendLine("      (fc3): ${linear(blk.ff.fc3)}")
        appendLine("    )")
        appendLine("    (input_layernorm): RMSNorm(${cfg.embDim})")
        appendLine("    (post_attention_layernorm): RMSNorm(${cfg.embDim})")
        appendLine("    (pre_feedforward_layernorm): RMSNorm(${cfg.embDim})")
        appendLine("    (post_feedforward_layernorm): RMSNorm(${cfg.embDim})")
        appendLine("  )")
        appendLine("  (final_norm): RMSNorm(${cfg.embDim})")
        appendLine("  (out_head): ${linear(outHead)}")
        append(")")
    }
}

println(model.summary())

Gemma3Model(
  (tok_emb): Embedding(262144, 640)
  (blocks): 18 x TransformerBlock(
    (att): GroupedQueryAttention(
      (W_query): Linear(inFeatures=640, outFeatures=1024, bias=false)
      (W_key): Linear(inFeatures=640, outFeatures=256, bias=false)
      (W_value): Linear(inFeatures=640, outFeatures=256, bias=false)
      (out_proj): Linear(inFeatures=1024, outFeatures=640, bias=false)
      (q_norm): RMSNorm(256)
      (k_norm): RMSNorm(256)
    )
    (ff): FeedForward(
      (fc1): Linear(inFeatures=640, outFeatures=2048, bias=false)
      (fc2): Linear(inFeatures=640, outFeatures=2048, bias=false)
      (fc3): Linear(inFeatures=2048, outFeatures=640, bias=false)
    )
    (input_layernorm): RMSNorm(640)
    (post_attention_layernorm): RMSNorm(640)
    (pre_feedforward_layernorm): RMSNorm(640)
    (post_feedforward_layernorm): RMSNorm(640)
  )
  (final_norm): RMSNorm(640)
  (out_head): Linear(inFeatures=640, outFeatures=262144, bias=false)
)


- A quick check that the forward pass works before continuing:

In [11]:
val testLogits = model.forward(intArrayOf(1, 2, 3))
println(testLogits.shape)

Shape: Dimensions = [1 x 3 x 262144], Size (Volume) = 786432


- Parameter counting. The out-head is tied to the token embedding after loading, so it is excluded from the unique count — these numbers match the PyTorch notebook exactly (435,870,336 / 268,098,176):

In [12]:
fun Gemma3Model.namedParameters(): List<Pair<String, FTensor>> = buildList {
    add("tok_emb.weight" to tokEmb)
    blocks.forEachIndexed { i, blk ->
        val p = "blocks.$i"
        add("$p.att.W_query.weight" to blk.att.wQuery.weight)
        add("$p.att.W_key.weight" to blk.att.wKey.weight)
        add("$p.att.W_value.weight" to blk.att.wValue.weight)
        add("$p.att.out_proj.weight" to blk.att.outProj.weight)
        blk.att.qNorm?.let { add("$p.att.q_norm.scale" to it.scale) }
        blk.att.kNorm?.let { add("$p.att.k_norm.scale" to it.scale) }
        add("$p.ff.fc1.weight" to blk.ff.fc1.weight)
        add("$p.ff.fc2.weight" to blk.ff.fc2.weight)
        add("$p.ff.fc3.weight" to blk.ff.fc3.weight)
        add("$p.input_layernorm.scale" to blk.inputLayernorm.scale)
        add("$p.post_attention_layernorm.scale" to blk.postAttentionLayernorm.scale)
        add("$p.pre_feedforward_layernorm.scale" to blk.preFeedforwardLayernorm.scale)
        add("$p.post_feedforward_layernorm.scale" to blk.postFeedforwardLayernorm.scale)
    }
    add("final_norm.scale" to finalNorm.scale)
    add("out_head.weight" to outHead.weight)
}

val totalParams = model.namedParameters().sumOf { it.second.volume.toLong() }
val uniqueParams = totalParams - model.outHead.weight.volume
println("Total number of parameters: %,d".format(totalParams))
println("Total number of unique parameters: %,d".format(uniqueParams))
println("FP32 memory for unique parameters: %.2f GB".format(uniqueParams * 4.0 / (1 shl 30)))

Total number of parameters: 435,870,336
Total number of unique parameters: 268,098,176
FP32 memory for unique parameters: 1.00 GB


&nbsp;
## Load the pretrained weights

The checkpoint ships as bf16 safetensors; `SafeTensorsParametersLoader` streams it tensor-by-tensor and dequantizes to FP32 by default. Because our `Linear` keeps the HF `[out, in]` layout, every tensor drops straight into place — the only mapping is from HF names to our fields, and the missing `lm_head.weight` triggers weight tying, exactly like the original.

> Google requires accepting the Gemma license before downloading from `google/gemma-3-270m-it` — set the `HF_TOKEN` environment variable after accepting. The `unsloth/` mirrors host the identical files ungated, which is what this notebook defaults to.

In [13]:
val repoId = if (useInstructModel) "unsloth/gemma-3-270m-it" else "unsloth/gemma-3-270m"
// Official (gated — accept the license on huggingface.co and export HF_TOKEN):
// val repoId = if (useInstructModel) "google/gemma-3-270m-it" else "google/gemma-3-270m"

val resolver = JvmDataSourceResolver(useEnvironmentHuggingFaceToken = true)

fun download(file: String): String = blocking {
    val artifact = resolver.resolve(DataSourceRequest("hf://$repoId/$file"))
    println("$file -> ${artifact.localPath} (cacheHit=${artifact.cacheHit})")
    artifact.localPath ?: error("No local path for $file")
}

val weightsPath = download("model.safetensors")
val tokenizerPath = download("tokenizer.json")

model.safetensors -> /Users/A9973957/.cache/skainet/data/huggingface-8ada7195-model.safetensors (cacheHit=true)
tokenizer.json -> /Users/A9973957/.cache/skainet/data/huggingface-40e1858a-tokenizer.json (cacheHit=true)


In [14]:
val params = HashMap<String, FTensor>()
blocking {
    SafeTensorsParametersLoader(sourceProvider = { createRandomAccessSource(weightsPath)!! })
        .load<FP32, Float>(ctx, FP32::class) { name, tensor -> params[name] = tensor }
}
println("Loaded ${params.size} tensors")

Loaded 236 tensors


In [15]:
fun loadWeightsIntoGemma(model: Gemma3Model, params: Map<String, FTensor>) {
    fun assign(expected: Shape, name: String): FTensor {
        val right = params[name] ?: error("Missing tensor '$name'")
        require(expected.dimensions.contentEquals(right.shape.dimensions)) {
            "Shape mismatch in '$name'. Left: $expected, Right: ${right.shape}"
        }
        return right
    }

    model.tokEmb = assign(model.tokEmb.shape, "model.embed_tokens.weight")

    model.blocks.forEachIndexed { l, block ->
        val att = block.att
        att.wQuery.weight = assign(att.wQuery.weight.shape, "model.layers.$l.self_attn.q_proj.weight")
        att.wKey.weight = assign(att.wKey.weight.shape, "model.layers.$l.self_attn.k_proj.weight")
        att.wValue.weight = assign(att.wValue.weight.shape, "model.layers.$l.self_attn.v_proj.weight")
        att.outProj.weight = assign(att.outProj.weight.shape, "model.layers.$l.self_attn.o_proj.weight")
        att.qNorm?.let { it.scale = assign(it.scale.shape, "model.layers.$l.self_attn.q_norm.weight") }
        att.kNorm?.let { it.scale = assign(it.scale.shape, "model.layers.$l.self_attn.k_norm.weight") }
        block.ff.fc1.weight = assign(block.ff.fc1.weight.shape, "model.layers.$l.mlp.gate_proj.weight")
        block.ff.fc2.weight = assign(block.ff.fc2.weight.shape, "model.layers.$l.mlp.up_proj.weight")
        block.ff.fc3.weight = assign(block.ff.fc3.weight.shape, "model.layers.$l.mlp.down_proj.weight")
        block.inputLayernorm.scale = assign(block.inputLayernorm.scale.shape, "model.layers.$l.input_layernorm.weight")
        block.postAttentionLayernorm.scale =
            assign(block.postAttentionLayernorm.scale.shape, "model.layers.$l.post_attention_layernorm.weight")
        block.preFeedforwardLayernorm.scale =
            assign(block.preFeedforwardLayernorm.scale.shape, "model.layers.$l.pre_feedforward_layernorm.weight")
        block.postFeedforwardLayernorm.scale =
            assign(block.postFeedforwardLayernorm.scale.shape, "model.layers.$l.post_feedforward_layernorm.weight")
    }

    model.finalNorm.scale = assign(model.finalNorm.scale.shape, "model.norm.weight")

    if ("lm_head.weight" in params) {
        model.outHead.weight = assign(model.outHead.weight.shape, "lm_head.weight")
    } else {
        model.outHead.weight = model.tokEmb // weight tying
        println("Model uses weight tying.")
    }
}

loadWeightsIntoGemma(model, params)

Model uses weight tying.


&nbsp;
## Tokenizer and chat template

Gemma's `tokenizer.json` is SentencePiece-style BPE with byte fallback; `TextFrontEnds.fromTokenizerJson` dispatches to the matching implementation and wraps it so special tokens like `<start_of_turn>` encode atomically. The chat template is simple enough to write by hand — note that `assistant` becomes `model`, and that we prepend `<bos>` ourselves (the id-level equivalent of the Python tokenizer's `add_bos_token = True`).

In [16]:
val tokenizer = TextFrontEnds.fromTokenizerJson(File(tokenizerPath).readText())

val bosTokenId = tokenizer.vocabulary.tokenToId("<bos>")!!
val eosTokenId = tokenizer.vocabulary.tokenToId("<eos>")!!
val endOfTurnTokenId = tokenizer.vocabulary.tokenToId("<end_of_turn>")!!

fun applyChatTemplate(userText: String): String =
    "<start_of_turn>user\n$userText<end_of_turn>\n<start_of_turn>model\n"

In [17]:
val prompt = "Give me a short introduction to large language models."

val inputTokenIds = intArrayOf(bosTokenId) + tokenizer.encode(applyChatTemplate(prompt))
tokenizer.decode(inputTokenIds)

<bos><start_of_turn>user
Give me a short introduction to large language models.<end_of_turn>
<start_of_turn>model


&nbsp;
## Streaming generation through the KV cache

A Kotlin `Sequence` is the natural twin of the Python generator: prime the cache with the whole prompt once, then feed **one token per step**. Greedy decoding — the argmax over the last position's logits.

In [18]:
fun generateTextStream(
    model: Gemma3Model,
    tokenIds: IntArray,
    maxNewTokens: Int,
    eosTokenId: Int? = null,
): Sequence<Int> = sequence {
    val cache = KVCache(model.cfg.nLayers)
    model.resetKvCache()

    // Prime the cache with the initial context
    var logits = model.forward(tokenIds, cache)

    for (step in 0 until maxNewTokens) {
        val lastLogits = logits.narrow(dim = 1, start = logits.shape[1] - 1, length = 1)
        val nextToken = logits.ops.argMax(lastLogits, dim = 2).data.get(0, 0).toInt()

        if (eosTokenId != null && nextToken == eosTokenId) break
        yield(nextToken)

        // Feed only the new token; the cache carries the history
        if (step + 1 < maxNewTokens) {
            logits = model.forward(intArrayOf(nextToken), cache)
        }
    }
}

In [27]:
var generated = 0
val t0 = System.currentTimeMillis()

for (token in generateTextStream(model, inputTokenIds, maxNewTokens = 500, eosTokenId = endOfTurnTokenId)) {
    print(tokenizer.decodeToken(token))
    System.out.flush()
    generated++
}

val dt = (System.currentTimeMillis() - t0) / 1000.0
println("\n\n$generated tokens in %.1f s (%.2f tok/s)".format(dt, generated / dt))

Large language models (LLMs) are sophisticated artificial intelligence systems that can understand, generate, and manipulate human language. They are trained on massive amounts of text data to learn patterns and relationships within language, enabling them to perform a wide range of tasks, from writing articles and answering questions to translating languages and summarizing information.


64 tokens in 7.9 s (8.11 tok/s)


&nbsp;
## What's next

- **Verify the KV cache**: run `model.forward(inputTokenIds)` without a cache and compare the last position's logits against the cached run — on this CPU backend they match bit-for-bit.
- **Compress the cache**: SKaiNET ships TurboQuant (`KvCacheStore.turboQuant(...)`) for 2-8 bit KV-cache encodings — a natural extension of the `LayerCache` here.
- **The production route**: the [SKaiNET-transformers](https://github.com/SKaiNET/SKaiNET-transformers) repository provides these layers (and GGUF loading, paged caches, samplers) as a library — this notebook is the "how it works", that repo is the "how you'd ship it".
- Based on Sebastian Raschka's [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) — see his [Build a Large Language Model (From Scratch)](https://www.manning.com/books/build-a-large-language-model-from-scratch) book for the comprehensive guide.